# NOAA GoMOFS in Icechunk: surface temperature map and a buoy time series

Opens the virtual Icechunk store `timeseries-3d` (GoMOFS `fields` files, 3-hourly, one `time`
dimension of valid times) straight from Source Cooperative, and makes two plots:

1. a map of the latest nowcast sea-surface temperature, and
2. a time series at NERACOOS buoy **B01** (Western Maine Shelf) over the last month of the record,
   which ends with the newest 72 h forecast.

The store holds only references into NOAA's `s3://noaa-nos-ofs-pds` bucket; no data is copied, so reads
fetch bytes from NOAA. Nothing here needs credentials.

In [ ]:
# %pip install "icechunk>=2.1" xarray zarr dask hvplot holoviews bokeh datashader "geoviews>=1.16" cartopy
# icechunk 2.x needs Python >= 3.12.
# geoviews < 1.16 projects curvilinear (2-D lon/lat) grids with a ~23 km northward shift: use 1.16 or newer.

In [ ]:
import icechunk
import numpy as np
import pandas as pd
import xarray as xr
import hvplot.xarray  # noqa: F401  (registers the .hvplot accessor)
import holoviews as hv

STORE = "https://data.source.coop/ocean-icechunks/noaa-gomofs/timeseries-3d"
NOAA = "s3://noaa-nos-ofs-pds/"

storage = icechunk.http_storage(STORE)
repo = icechunk.Repository.open(
    storage,
    authorize_virtual_chunk_access=icechunk.containers_credentials(
        {NOAA: icechunk.s3_anonymous_credentials()}
    ),
)
# chunks={} keeps the stored chunking as dask chunks, so selections stay lazy
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
ds

`time` is the valid time. Each cycle's nowcast files tile the timeline; the **newest** cycle also contributes its
72 h forecast, which every update replaces. The root attribute `last_cycle` is the newest cycle, so everything
after it is forecast.

In [ ]:
last_cycle = pd.Timestamp(ds.attrs["last_cycle"])
print("steps:", ds.sizes["time"], "| first:", ds.time.values[0], "| newest cycle:", last_cycle,
      "| last (forecast):", ds.time.values[-1])

## Surface temperature now

The latest nowcast field (`s_rho=-1` is the surface layer). The grid is curvilinear, so the plot uses the 2-D `lon_rho`/`lat_rho` coordinates.

In [ ]:
sst = ds.temp.sel(time=last_cycle).isel(s_rho=-1).load()
sst.hvplot.quadmesh(
    x="lon_rho", y="lat_rho", rasterize=True, geo=True, tiles="OSM", cmap="turbo", clabel="°C",
    frame_width=700, title=f"GoMOFS surface temperature, {last_cycle:%Y-%m-%d %H}Z",
)

## NERACOOS buoy B01

B01 (Western Maine Shelf) is at about 43.18°N, 70.43°W. Find the nearest ocean grid point.

In [ ]:
B01_LON, B01_LAT = -70.43, 43.18

lon, lat, mask = ds.lon_rho.load(), ds.lat_rho.load(), ds.mask_rho.load()
dist_km = 111.2 * np.hypot(lat - B01_LAT, (lon - B01_LON) * np.cos(np.deg2rad(B01_LAT)))
dist_km = dist_km.where(mask == 1)                       # ocean points only
j, i = np.unravel_index(np.nanargmin(dist_km.values), dist_km.shape)
print(f"nearest ocean grid point: eta_rho={j}, xi_rho={i} at "
      f"{float(lat[j, i]):.3f}N, {float(lon[j, i]):.3f}E, {float(dist_km[j, i]):.2f} km away; "
      f"depth {float(ds.h[j, i]):.0f} m")

## Surface temperature at B01, last month of the record

Thirty days up to the newest cycle, then the 72 h forecast. This reads about 240 steps, so it takes a minute or two.

In [ ]:
window = ds.temp.isel(s_rho=-1, eta_rho=j, xi_rho=i).sel(time=slice(last_cycle - pd.Timedelta(days=30), None))
ts = window.compute()
nowcast, forecast = ts.sel(time=slice(None, last_cycle)), ts.sel(time=slice(last_cycle, None))

(nowcast.hvplot.line(label="nowcast", color="#1f77b4") * forecast.hvplot.line(label="forecast", color="#ff7f0e", line_dash="dashed")
 * hv.VLine(last_cycle).opts(color="gray", line_dash="dotted")).opts(
    title="GoMOFS surface temperature at B01 (nearest grid point)", ylabel="°C", xlabel="valid time (UTC)",
    frame_width=700, legend_position="top_left")

## Notes

- The oldest part of the store moves: NOAA deletes old GoMOFS files about a day a day, and the store is rebuilt
  now and then with a later start. Check `ds.time.values[0]` rather than assuming a fixed start.
- `timeseries-2d` holds the hourly surface (`2ds`) fields with the same layout. Replace `timeseries-3d` in `STORE`;
  the surface fields have no `s_rho` dimension, so drop `.isel(s_rho=-1)`.